In [ ]:
from os import environ

environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
from torch.nn import *
from sklearn.datasets import make_classification
import numpy as np
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n):
    if n < 9:
        return np.unpackbits(a.view("B")[::4, None], 1)[:, -n:]
    return np.unpackbits(a[:, None].view("B")[:, :2], 1)[
        :, [*range(8), *range(24 - n, 16)]
    ]


class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.view(-1, x.size(-1))).view(x.shape)


def los(x, y):
    return functional.binary_cross_entropy_with_logits(x, y.broadcast_to(x.shape))


def acc(x, y):
    x = x.signbit() != y
    return x.sum() / x.numel()


def dat(n, l, b=1):
    return torch.tensor(
        np.stack(
            [
                np.concatenate((a[0].astype("f"), unpack(a[1], n).astype("f")), 1)
                for a in range(b)
                for a in [
                    make_classification(
                        l,
                        n,
                        n_informative=n,
                        n_redundant=0,
                        n_classes=2 ** n,
                        n_clusters_per_class=1,
                    )
                ]
            ],
            1,
        )
    )


def fast_eval(m, d, x, y):
    i = torch.randint(len(d), d.shape[:1]) % torch.arange(1, len(d) + 1)
    return torch.cat((d, x[i]), 2), y[i]


def slow_eval(m, d, x, y):
    i, I = torch.tril_indices(*d.shape[:1] * 2)
    return m(torch.cat((d[i], x[I]), 2)), y[I]


def full_eval(m, d, x, y):
    return (
        m(
            torch.cat(
                (
                    d[:, None].broadcast_to(len(d), *d.shape),
                    x.broadcast_to(len(d), *x.shape),
                ),
                3,
            )
        ),
        y,
    )


def last_eval(m, d, x, y):
    w, W = m[0].weight.T.split((d.size(2), x.size(2)), 0)
    return m[1:](d @ w + x @ W), y

In [ ]:
n = 4
m = int(n * 2 ** (n / 2)) * 2
M = GRU(n * 2, m)
m = Sequential(
    Linear(m + n, m * 4, 0),
    BatchNorm(m * 4, track_running_stats=0),
    Hardswish(1),
    Linear(m * 4, n),
)
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
# M, m, o = torch.load("nnTrainerEvaler-Copy4.model.pk")
s = torch.cuda.amp.GradScaler()
sum(i.numel() for i in o.param_groups[0]["params"])

In [ ]:
for i in range(20):
    for i in range(100):
        o.zero_grad()
        with torch.cuda.amp.autocast():
            a = dat(n, 256 * 4)
            a = slow_eval(m, M(a)[0], a[..., :n], a[..., n:])
            i = los(*a)
        s.scale(i).backward()
        s.step(o)
        s.update()
        # i.backward()
        # o.step()
    print(i, acc(*a))

In [ ]:
for i in range(20):
    for i in range(100):
        o.zero_grad()
        with torch.cuda.amp.autocast():
            a = dat(n, 256 * 4)
            a = slow_eval(m, M(a)[0], a[..., :n], a[..., n:])
            i = los(*a)
        s.scale(i).backward()
        s.step(o)
        s.update()
        # i.backward()
        # o.step()
    print(i, acc(*a))